# Notebook 05 — PCA from First Principles

**Module:** 04 — Linear Algebra  
**Notebook:** 05 of 10 — Track A core  
**Prerequisites:** NB04  
**Time estimate:** 90 minutes

> **Track A note:** 'Plot a PCA of these samples and explain what you see' is a
> first-day task in most bioinformatics roles. Being able to implement PCA from
> scratch and interpret the result separates strong from average candidates.

---
## Step 1 — Motivation

An RNA-seq experiment produces 20,000 numbers per sample. You can't plot 20,000
axes. PCA compresses those 20,000 numbers into 2–3 numbers that capture the main
variation — without losing the most important information. This notebook derives
PCA from the variance-maximization objective and implements it from scratch.

---
## Step 2 — Intuition

Imagine each sample is a point in 20,000-dimensional gene space. PCA finds the
best low-dimensional plane through those points — 'best' meaning: the projection
onto the plane retains the maximum spread (variance) of the points.

The first PC is the direction of maximum variance. The second PC is the direction
of maximum remaining variance, constrained to be perpendicular to PC1. And so on.

---
## Step 3 — Biological Background

**What PCA reveals in expression data:**
- **PC1** often captures the dominant source of variation: if there are two treatment
  groups, PC1 usually separates them. If there's a batch effect, PC1 might capture
  that instead — a sign to regress out batch before analysis.
- **PC2** captures the second source: perhaps cell cycle stage, sex, or a secondary
  condition.
- **Gene loadings** (coefficients in the linear combination defining each PC) tell
  you which genes drive the variation in each PC.

**Practical PCA checklist (Track A):**
1. Log-normalize expression data (log2CPM or similar)
2. Filter: top 5,000 most variable genes (by variance)
3. Center (subtract gene means across samples)
4. Run PCA on the centered data
5. Color PC scatter by metadata (condition, batch, etc.)
6. Inspect loadings to identify driving genes

---
## Step 4 — Mathematical Explanation

**PCA objective:** find unit vector $\mathbf{w}_1$ maximizing variance of projections:
$$\mathbf{w}_1 = \arg\max_{\|\mathbf{w}\|=1} \text{Var}(\mathbf{X}\mathbf{w}) = \arg\max_{\|\mathbf{w}\|=1} \mathbf{w}^\top \mathbf{C} \mathbf{w}$$

where $\mathbf{C} = \frac{1}{n-1}\mathbf{X}^\top \mathbf{X}$ is the sample covariance matrix
(with $\mathbf{X}$ centered, genes as columns, samples as rows).

**Solution:** $\mathbf{w}_1$ is the eigenvector of $\mathbf{C}$ with the largest eigenvalue.
This follows from the method of Lagrange multipliers:
$$\nabla_{\mathbf{w}}[\mathbf{w}^\top \mathbf{C}\mathbf{w} - \lambda(\mathbf{w}^\top\mathbf{w} - 1)] = 0
\implies \mathbf{C}\mathbf{w} = \lambda\mathbf{w}$$

Maximizing $\mathbf{w}^\top \mathbf{C}\mathbf{w} = \lambda\mathbf{w}^\top\mathbf{w} = \lambda$
→ pick the eigenvector with the largest $\lambda$.

**Algorithm (from scratch):**
1. Center: $\tilde{\mathbf{X}} = \mathbf{X} - \bar{\mathbf{X}}$
2. Covariance: $\mathbf{C} = \frac{1}{n-1}\tilde{\mathbf{X}}^\top \tilde{\mathbf{X}}$
3. Eigendecomposition: $\mathbf{C} = \mathbf{Q}\Lambda\mathbf{Q}^\top$
4. Sort by $\lambda$ descending
5. PC scores: $\mathbf{Z} = \tilde{\mathbf{X}} \mathbf{Q}_k$ (project onto top $k$ eigenvectors)

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA as sklearn_PCA

In [ ]:
# Cell 6.1 — PCA from scratch: full implementation
def pca_from_scratch(X: np.ndarray, n_components: int) -> dict:
    """
    Principal Component Analysis via eigendecomposition of covariance matrix.

    Parameters
    ----------
    X : (n_samples, n_features) data matrix — will be centered internally
    n_components : number of PCs to return

    Returns
    -------
    dict with: scores, loadings, explained_variance_ratio, eigenvalues
    """
    X = np.asarray(X, dtype=float)
    n_samples, n_features = X.shape

    # Step 1: Center
    X_centered = X - X.mean(axis=0)

    # Step 2: Covariance matrix (features × features)
    # For n_features >> n_samples, work with the Gram matrix instead (NB06 covers SVD trick)
    C = (X_centered.T @ X_centered) / (n_samples - 1)   # (features, features)

    # Step 3: Eigendecomposition (eigh for symmetric — guaranteed real, sorted)
    eigenvalues, eigenvectors = np.linalg.eigh(C)

    # Step 4: Sort descending
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues  = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]   # columns are eigenvectors

    # Step 5: Select top k
    loadings = eigenvectors[:, :n_components]   # (features, k)

    # Step 6: Project
    scores = X_centered @ loadings              # (n_samples, k)

    explained_var_ratio = eigenvalues[:n_components] / eigenvalues.sum()

    return dict(scores=scores, loadings=loadings,
                explained_variance_ratio=explained_var_ratio,
                eigenvalues=eigenvalues,
                X_centered=X_centered)

# Build synthetic data: 3 cell types
rng = np.random.default_rng(42)
n_samples, n_genes = 90, 200
cell_type = np.repeat([0, 1, 2], 30)
type_profiles = rng.normal(0, 3, (3, n_genes))
X = (type_profiles[cell_type] + rng.normal(0, 0.8, (n_samples, n_genes)))

res = pca_from_scratch(X, n_components=5)
print(f"PC scores shape: {res['scores'].shape}")
print(f"Explained variance: {res['explained_variance_ratio'].round(4)}")
print(f"Cumulative (top 3): {res['explained_variance_ratio'][:3].sum():.2%}")

In [ ]:
# Cell 6.2 — Validate against sklearn PCA
sk = sklearn_PCA(n_components=5, random_state=42)
sk.fit(X)

print("Explained variance comparison:")
print(f"  Scratch: {res['explained_variance_ratio'].round(6)}")
print(f"  sklearn: {sk.explained_variance_ratio_.round(6)}")
print(f"  Match:   {np.allclose(res['explained_variance_ratio'], sk.explained_variance_ratio_, atol=1e-6)}")

# PC scores match up to sign flip (PCs are defined up to sign)
sk_scores = sk.transform(X)
for i in range(2):
    corr = np.corrcoef(res['scores'][:, i], sk_scores[:, i])[0, 1]
    print(f"  PC{i+1} score correlation: {abs(corr):.8f}  (should be ~1.0)")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — PCA results: scatter, scree, loadings heatmap
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['steelblue', 'tomato', 'green']

# Panel 1: PC1 vs PC2 scatter
ax = axes[0]
for ct in range(3):
    mask = cell_type == ct
    ax.scatter(res['scores'][mask, 0], res['scores'][mask, 1], s=25, alpha=0.8,
               color=colors[ct], label=f'Type {ct}')
ax.set_xlabel(f"PC1 ({res['explained_variance_ratio'][0]:.1%})")
ax.set_ylabel(f"PC2 ({res['explained_variance_ratio'][1]:.1%})")
ax.set_title('PCA: 3 cell types clearly separated')
ax.legend(fontsize=8)

# Panel 2: Scree plot
ax = axes[1]
k_show = min(20, len(res['eigenvalues']))
evr = res['eigenvalues'][:k_show] / res['eigenvalues'].sum()
ax.bar(range(1, k_show+1), evr, color='steelblue', alpha=0.8)
ax.plot(range(1, k_show+1), np.cumsum(evr), 'tomato', lw=2, marker='o', ms=4,
        label='Cumulative')
ax.axhline(0.9, color='gray', ls='--', lw=1, label='90%')
ax.set_xlabel('PC'); ax.set_ylabel('Explained variance')
ax.set_title('Scree plot')
ax.legend(fontsize=8)

# Panel 3: Top gene loadings for PC1 and PC2
ax = axes[2]
top20 = np.argsort(np.abs(res['loadings'][:, 0]))[-20:]
ax.barh(range(20), res['loadings'][top20, 0], color='steelblue', alpha=0.8, label='PC1')
ax.barh(range(20), res['loadings'][top20, 1], color='tomato', alpha=0.6, label='PC2')
ax.set_ylabel('Gene index'); ax.set_xlabel('Loading')
ax.set_title('Top 20 gene loadings on PC1/PC2')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `pca_from_scratch()` without looking. Test it on `X` above and verify
   your explained variance ratios match sklearn to 6 decimal places.
2. What happens if you omit the centering step (Step 1)? Redo the analysis without
   centering and compare the PC1 scatter plot. Explain the difference.
3. The loadings matrix has shape (n_features, k). What does loading $[g, 1]$ mean
   biologically (the loading of gene $g$ on PC1)?
4. Add an artificial batch effect: multiply all samples 0–29 by a scalar 1.5 before
   running PCA. Which PC does the batch effect appear on? How would you detect it
   in a real experiment?

---
## Quiz — Active Recall (Track A critical)

1. Write the PCA objective function (variance maximization form).
2. Why is the solution to the PCA objective the leading eigenvector of the covariance matrix?
3. Walk through the 6 steps of `pca_from_scratch()` from memory.
4. PC scores are defined up to sign flip. What does this mean? Is it a bug?
5. What does the scree plot show, and how do you choose how many PCs to keep?

---
## Reflection

**Date completed:** ____________________

> *[Could you implement PCA from scratch in a 30-minute interview? What part would you reach for first?]*

---
*Next: `06_svd_and_connection_to_pca.ipynb`*